In [1]:
import torch
from diffusion.approaches.matching.alphas_betas import LinearAlpha, LinearBeta
from diffusion.approaches.matching.prob_paths import GaussianCondProbPath
from diffusion.approaches.matching.flow_trainer import FlowTrainer
from diffusion.data.mnist_sampleable import MNISTSampleable
from diffusion.architectures.res_unet import ResUnet
from diffusion.architectures.classifier.mnist_classifier import SimpleCNN

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
sampeable = MNISTSampleable(train=True)

path = GaussianCondProbPath(
    p_data=sampeable,
    p_simple_shape=(1, 32, 32),
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

backbone = ResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

encoder = SimpleCNN(sampeable.num_classes).to(device)
encoder.load_state_dict(torch.load("./models/classifier.pt"))

trainer = FlowTrainer(
    path=path,
    backbone=backbone,
    encoder=encoder,
    num_classes=sampeable.num_classes,
)

In [4]:
trainer.train(
    num_epochs=5,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
)

2025-10-10 13:22:56,378 - flow-matching - INFO - Training model with size: 2.734 MiB
100%|██████████| 9/9 [00:13<00:00,  1.54s/it]
2025-10-10 13:24:49,765 - flow-matching - INFO - ['kid: 19316.580859', 'precision: 0.892700', 'recall: 0.790400', 'f1: 0.837271']
100%|██████████| 9/9 [00:13<00:00,  1.54s/it]
2025-10-10 13:26:42,278 - flow-matching - INFO - ['kid: 16666.297852', 'precision: 0.780900', 'recall: 0.921100', 'f1: 0.844171']
100%|██████████| 9/9 [00:13<00:00,  1.54s/it]
2025-10-10 13:28:34,406 - flow-matching - INFO - ['kid: 42319.129687', 'precision: 0.709200', 'recall: 0.909400', 'f1: 0.795336']
100%|██████████| 9/9 [00:13<00:00,  1.55s/it]
2025-10-10 13:30:26,983 - flow-matching - INFO - ['kid: 138630.098438', 'precision: 0.634100', 'recall: 0.879800', 'f1: 0.734295']
Epoch 4/5:  19%|█▉        | 96/500 [00:07<00:30, 13.10it/s, loss=0.141817]


KeyboardInterrupt: 

In [6]:
torch.save(backbone.state_dict(), "./models/backbone_flow.pt")